# Verify lmz's GPU decoder on this card

**Set the runtime first:** Runtime → Change runtime type → **GPU**.
Then Runtime → Run all. It takes under a minute.

## What this is for

lmz's CUDA decoder compiles from sm_75 to sm_121 and is clean under
`compute-sanitizer`, but compiling is not running. Counting `cp.async`
instructions in the generated code shows the architectures fall into two
kinds, and which silicon has actually executed each:

| | `LDGSTS` (real `cp.async`) | run on real hardware |
|---|---|---|
| sm_75, Turing — a Colab T4 | **0** | **yes** — 30/30 byte-identical |
| **sm_80 / 86 / 89** — A100, A10G, L4, RTX 30/40 | 38 | **not yet — this is the gap** |
| sm_90 / 120 — H100, RTX 50 | 41 | **yes** — 936 MB byte-identical |

Turing has no `cp.async` instruction, so the intrinsic falls back to a
synchronous copy and sm_75 is genuinely different generated code. That was the
open question, and a T4 answered it: **verdict OK, thirty shapes.**

So a T4 run now just adds a second data point — still worth having, and still
one click. The **more useful** run today is an Ampere or Ada: A100, A10G, L4,
RTX 3090 or 4090. Those share the verified 38-`LDGSTS` path and have been
JIT-run and sanitized, but no such card has executed lmz's decoder. If Colab
offers you an **L4 or A100** under Change runtime type, prefer it.

Nothing here needs a data file or a login: the streams are built by lmz's own
encoder and checked against lmz's own decoder, so the oracle travels with the
question.


In [ ]:
!pip install -q lmzip
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv,noheader
!nvcc --version | tail -2


In [ ]:
!python -m lmz doctor --gpu-verify


## Please paste the block above into an issue

https://github.com/FanxinSun/lmz/issues

**A pass is evidence too.** A `verdict OK` from an Ampere or Ada card would be
the first from silicon in the middle of the range — the only part of the
matrix no hardware has executed. A `MISMATCH` is more valuable still: it means
the decoder correctly refused to trust itself on hardware it had never seen,
which is what it was built to do.

Either way lmz keeps working. The GPU decoder is optional in every direction:
no CUDA is installed by `pip install lmzip`, and a device that disagrees with
the CPU decoder is never used.

**Known-good, for comparison** — a stock Colab T4 runtime:

```
Tesla T4, 7.5, 580.82.07
Cuda compilation tools, release 12.8, V12.8.93

lmz 1.1.3 GPU verification
  device   Tesla T4 sm_75 40 SMs
  shapes   30 decoded byte-identically to the CPU decoder
  verdict  OK
```

The `batch` line the command also prints is a host round trip and includes
PCIe in both directions, so it is not the kernel's own rate — the resident
numbers are in
[docs/gpu-residency-handover.md](https://github.com/FanxinSun/lmz/blob/main/docs/gpu-residency-handover.md).
